# 안전귀가Navi — Day 5 AI 리포트 생성

- 입력: `scoring.score_signal()` 출력 JSON
- 출력: 자연어 리포트 + 면책 문구
- LLM 없으면 템플릿 fallback (이번 검증은 fallback로)

## 검증 항목
1. 템플릿 fallback 정상 동작 (모든 샘플 좌표)
2. 금지 표현 검출 함수 동작
3. (LLM 키 있을 때) 실제 OpenAI 호출
4. 등급별 다양한 리포트 톤

In [1]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path(r'C:\Users\pjhic\Projects\Seoul_bigdata\src')))
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import search, scoring, report

print(f'OpenAI 키 설정: {bool(report.OPENAI_KEY)}')
print(f'모델: {report.OPENAI_MODEL}')
print(f'금지 패턴 수: {len(report.FORBIDDEN_PATTERNS)}')

OpenAI 키 설정: False
모델: gpt-4o-mini
금지 패턴 수: 9


## 1. 금지 표현 검출 함수 테스트

In [2]:
test_cases = [
    ('이 주소 주변의 안전지원 인프라는 양호한 편입니다.',           False),
    ('이 지역은 위험합니다. 피해야 합니다.',                          True),
    ('범죄 가능성이 높은 동네입니다.',                                 True),
    ('여성에게 위험한 동네입니다.',                                   True),
    ('야간 조명 인프라가 부족하여 보완이 필요한 주소입니다.',         False),
    ('우범지역으로 분류됩니다.',                                       True),
    ('CCTV와 안심귀갓길 접근성이 양호합니다.',                        False),
]

for text, expected in test_cases:
    actual = report.has_forbidden(text)
    mark = '✓' if actual == expected else '✗'
    print(f'  {mark}  expected={expected}, got={actual}: {text}')

  ✓  expected=False, got=False: 이 주소 주변의 안전지원 인프라는 양호한 편입니다.
  ✓  expected=True, got=True: 이 지역은 위험합니다. 피해야 합니다.
  ✓  expected=True, got=True: 범죄 가능성이 높은 동네입니다.
  ✓  expected=True, got=True: 여성에게 위험한 동네입니다.
  ✓  expected=False, got=False: 야간 조명 인프라가 부족하여 보완이 필요한 주소입니다.
  ✓  expected=True, got=True: 우범지역으로 분류됩니다.
  ✓  expected=False, got=False: CCTV와 안심귀갓길 접근성이 양호합니다.


## 2. 샘플 좌표 → 점수 → 리포트
Day 4와 동일한 샘플로 리포트 생성.

In [3]:
samples = {
    '신촌역':         (37.5559, 126.9367),
    '관악구 신림동':   (37.4843, 126.9296),
    '강동구 천호동':   (37.5384, 127.1239),
    '잠실역':         (37.5133, 127.1000),
    '강남역':         (37.4979, 127.0276),
}

for name, (lat, lon) in samples.items():
    sig = search.analyze_lat_lon(lat, lon)
    sig['address'] = name
    score = scoring.score_signal(sig, priority='balanced')
    rep = report.generate_report(score)

    print(f'\n=== {name} ({score["total_score"]}점, {score["grade"]}) ===')
    print(f'[source: {rep["source"]}]')
    print(rep['report'])
    print(f'\n📌 {rep["disclaimer"]}')
    print(f'금지 표현 포함: {rep["has_forbidden"]}')


=== 신촌역 (88점, 매우 양호) ===
[source: template]
이 주소의 종합 안심 주거환경 점수는 88점으로 매우 양호 등급에 해당합니다. 감시 인프라 항목이 100점으로 가장 양호하며, 반경 300m 내 CCTV 46개, 가장 가까운 CCTV는 28m 거리에 있습니다. 반면 야간 조명 항목은 70점으로, 반경 100m 내 가로등이 1개로 야간 조명 인프라가 보완이 필요합니다. 주변 안전지원 인프라가 전반적으로 양호하므로 일반적인 야간 귀가 주의사항만 따르면 됩니다.

📌 본 결과는 공공데이터 기반 안전지원 인프라 분석이며, 실제 범죄 발생 가능성을 예측하지 않습니다.
금지 표현 포함: False

=== 관악구 신림동 (78점, 양호) ===
[source: template]
이 주소의 종합 안심 주거환경 점수는 78점으로 양호 등급에 해당합니다. 감시 인프라 항목이 100점으로 가장 양호하며, 반경 300m 내 CCTV 43개, 가장 가까운 CCTV는 30m 거리에 있습니다. 반면 야간 조명 항목은 42점으로, 반경 100m 내 가로등이 0개로 야간 조명 인프라가 보완이 필요합니다. 집 주변 조명 인프라가 상대적으로 낮게 나타나, 밤 시간대에는 조명이 많은 큰길 또는 안심귀갓길을 이용하는 것이 권장됩니다.

📌 본 결과는 공공데이터 기반 안전지원 인프라 분석이며, 실제 범죄 발생 가능성을 예측하지 않습니다.
금지 표현 포함: False

=== 강동구 천호동 (74점, 양호) ===
[source: template]
이 주소의 종합 안심 주거환경 점수는 74점으로 양호 등급에 해당합니다. 긴급 대응 항목이 100점으로 가장 양호하며, 반경 300m 내 안전비상벨 20개로 긴급 대응 인프라가 잘 갖추어져 있습니다. 반면 야간 조명 항목은 42점으로, 반경 100m 내 가로등이 0개로 야간 조명 인프라가 보완이 필요합니다. 집 주변 조명 인프라가 상대적으로 낮게 나타나, 밤 시간대에는 조명이 많은 큰길 또는 안심귀갓길을 이용하는 것이 권장됩


=== 강남역 (64점, 보통) ===
[source: template]
이 주소의 종합 안심 주거환경 점수는 64점으로 보통 등급에 해당합니다. 긴급 대응 항목이 94점으로 가장 양호하며, 반경 300m 내 안전비상벨 8개로 긴급 대응 인프라가 잘 갖추어져 있습니다. 반면 야간 조명 항목은 33점으로, 반경 100m 내 가로등이 0개로 야간 조명 인프라가 보완이 필요합니다. 집 주변 조명 인프라가 상대적으로 낮게 나타나, 밤 시간대에는 조명이 많은 큰길 또는 안심귀갓길을 이용하는 것이 권장됩니다.

📌 본 결과는 공공데이터 기반 안전지원 인프라 분석이며, 실제 범죄 발생 가능성을 예측하지 않습니다.
금지 표현 포함: False


## 3. priority별 리포트 톤 차이 (강남역)

In [4]:
sig = search.analyze_lat_lon(37.4979, 127.0276)
sig['address'] = '강남역 일대'

for priority in ['balanced', 'lighting', 'emergency']:
    score = scoring.score_signal(sig, priority=priority)
    rep = report.generate_report(score)
    print(f'\n--- priority={priority} (종합 {score["total_score"]}점, {score["grade"]}) ---')
    print(rep['report'])


--- priority=balanced (종합 64점, 보통) ---
이 주소의 종합 안심 주거환경 점수는 64점으로 보통 등급에 해당합니다. 긴급 대응 항목이 94점으로 가장 양호하며, 반경 300m 내 안전비상벨 8개로 긴급 대응 인프라가 잘 갖추어져 있습니다. 반면 야간 조명 항목은 33점으로, 반경 100m 내 가로등이 0개로 야간 조명 인프라가 보완이 필요합니다. 집 주변 조명 인프라가 상대적으로 낮게 나타나, 밤 시간대에는 조명이 많은 큰길 또는 안심귀갓길을 이용하는 것이 권장됩니다.

--- priority=lighting (종합 56점, 보통) ---
이 주소의 종합 안심 주거환경 점수는 56점으로 보통 등급에 해당합니다. 긴급 대응 항목이 94점으로 가장 양호하며, 반경 300m 내 안전비상벨 8개로 긴급 대응 인프라가 잘 갖추어져 있습니다. 반면 야간 조명 항목은 33점으로, 반경 100m 내 가로등이 0개로 야간 조명 인프라가 보완이 필요합니다. 집 주변 조명 인프라가 상대적으로 낮게 나타나, 밤 시간대에는 조명이 많은 큰길 또는 안심귀갓길을 이용하는 것이 권장됩니다.

--- priority=emergency (종합 71점, 양호) ---
이 주소의 종합 안심 주거환경 점수는 71점으로 양호 등급에 해당합니다. 긴급 대응 항목이 94점으로 가장 양호하며, 반경 300m 내 안전비상벨 8개로 긴급 대응 인프라가 잘 갖추어져 있습니다. 반면 야간 조명 항목은 33점으로, 반경 100m 내 가로등이 0개로 야간 조명 인프라가 보완이 필요합니다. 집 주변 조명 인프라가 상대적으로 낮게 나타나, 밤 시간대에는 조명이 많은 큰길 또는 안심귀갓길을 이용하는 것이 권장됩니다.


## 4. 전체 출력 구조 확인 — Day 6 UI에 전달될 페이로드

In [5]:
sig = search.analyze_lat_lon(37.5384, 127.1239)
sig['address'] = '서울특별시 강동구 천호동 (샘플)'
score = scoring.score_signal(sig, priority='balanced')
rep = report.generate_report(score)

# UI에 전달될 통합 페이로드
ui_payload = {
    'address':            score['address'],
    'lat':                score['lat'],
    'lon':                score['lon'],
    'total_score':        score['total_score'],
    'grade':              score['grade'],
    'one_line_summary':   score['one_line_summary'],
    'recommended_action': score['recommended_action'],
    'category_scores':    score['category_scores'],
    'ai_report':          rep['report'],
    'disclaimer':         rep['disclaimer'],
    'report_source':      rep['source'],
}
print(json.dumps(ui_payload, ensure_ascii=False, indent=2))

{
  "address": "서울특별시 강동구 천호동 (샘플)",
  "lat": 37.5384,
  "lon": 127.1239,
  "total_score": 74,
  "grade": "양호",
  "one_line_summary": "긴급 대응은 양호하지만 야간 조명은 보완이 필요한 주소입니다 (종합 74점).",
  "recommended_action": "집 주변 조명 인프라가 상대적으로 낮게 나타나, 밤 시간대에는 조명이 많은 큰길 또는 안심귀갓길을 이용하는 것이 권장됩니다.",
  "category_scores": {
    "surveillance": {
      "score": 77,
      "components": {
        "distance_score": 75,
        "density_score": 80
      },
      "evidence": {
        "nearest_m": 121.35338476956568,
        "count_300m": 10,
        "count_500m": 31
      },
      "active": true,
      "weight": 0.3,
      "name": "감시 인프라"
    },
    "lighting": {
      "score": 42,
      "components": {
        "distance_score": 60,
        "density_score": 15
      },
      "evidence": {
        "nearest_m": 238.70781598330785,
        "count_100m": 0,
        "count_300m": 1
      },
      "active": true,
      "weight": 0.25,
      "name": "야간 조명"
    },
    "emergency": {
      "score": 100,
      "components"